# Punto 4 — Pruebas para estudiar si las series son estacionarias

En el punto 2 se observaron gráficos. Aquí se agregan tres pruebas estadísticas: **ADF**, **Phillips–Perron (PP)** y **KPSS**. Se estudian las cuatro series originales y las diferencias propuestas.

## Ideas necesarias para leer los resultados

Una serie **estacionaria** mantiene una media y una variabilidad estables, y su relación con los valores anteriores depende del tiempo que los separa, no de la fecha. No significa que todos sus valores sean iguales ni que los días sean independientes.

Una **raíz unitaria** es una forma de falta de estacionariedad: un cambio inesperado puede tener un efecto permanente sobre el nivel de la serie. En esos casos, restar el valor anterior puede ayudar. Sin embargo, no toda serie no estacionaria tiene una raíz unitaria. Por ejemplo, una media que cambia de manera previsible con las estaciones requiere otro tipo de análisis.

Cada prueba parte de una **hipótesis nula (H₀)**, que es la afirmación que se pone a prueba. El **p-valor** ayuda a decidir si hay suficiente evidencia para rechazarla. Se utiliza el criterio del 5%:

- Si el p-valor es menor que 0,05, se rechaza H₀.
- Si es mayor o igual que 0,05, no se rechaza H₀. Esto significa que falta evidencia para descartarla, no que se haya demostrado que es verdadera.

## ¿Qué pregunta cada prueba?

| Prueba | Hipótesis de partida (H₀) | Lectura de un p-valor menor que 0,05 |
|---|---|---|
| ADF | La serie tiene una raíz unitaria. | Hay evidencia contra esa raíz unitaria. |
| Phillips–Perron (PP) | La serie tiene una raíz unitaria. | Hay evidencia contra esa raíz unitaria. |
| KPSS | La serie es estacionaria alrededor de una media o de una tendencia, según la opción elegida. | Hay evidencia contra esa estacionariedad. |

**Atención:** KPSS parte de la idea contraria a ADF y PP. Por eso no se interpretan todos los p-valores de la misma forma.

ADF y PP tratan de manera diferente la relación entre los errores de días cercanos. ADF agrega diferencias de días anteriores a su cálculo; PP ajusta el cálculo estadístico para tener en cuenta esa relación. Usar ambas permite comprobar si la conclusión cambia con el método.

## Opciones utilizadas

La opción principal es **`c`**, que incluye una constante: se evalúa el comportamiento alrededor de una media fija. Para los datos originales también se utiliza **`ct`**, que agrega una tendencia lineal: se evalúa si las fluctuaciones son estables alrededor de una línea que puede subir o bajar con el tiempo.

Los ajustes se seleccionan automáticamente: ADF usa AIC, un criterio que equilibra ajuste y complejidad; PP utiliza el ancho de banda automático de arch; KPSS utiliza `nlags='auto'`. El ancho de banda indica cuántas relaciones temporales se consideran en la corrección del cálculo y no es el orden de un modelo AR.

Si ADF y PP rechazan raíz unitaria y KPSS no rechaza estacionariedad, las pruebas son compatibles con estacionariedad bajo la opción utilizada. Si discrepan, el resultado se informa como **evidencia mixta**.

Estas pruebas no determinan por sí solas si hace falta una diferencia semanal o anual. Un ciclo estacional previsible puede existir aunque se rechace una raíz unitaria. Además, solo hay tres años disponibles para estudiar el patrón anual.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from statsmodels.tsa.stattools import acf, acovf, adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/bicicletas_rosario.csv').exists())
series = {}
for nombre, archivo, columna in [
    ('Viajes MBTB', 'bicicletas_rosario.csv', 'viajes'),
    ('Temperatura', 'temperatura_rosario.csv', 't2m_c'),
    ('Humedad', 'humedad_rosario.csv', 'rh2m_pct'),
    ('Precipitación', 'precipitacion_rosario.csv', 'precip_mm')]:
    df = pd.read_csv(ROOT / 'data' / archivo, parse_dates=['fecha'])
    assert not df.fecha.duplicated().any(), 'Fechas duplicadas'
    s = df.set_index('fecha')[columna].sort_index().asfreq('D')
    assert s.notna().all() and np.isfinite(s).all(), 'Revisar faltantes o valores no finitos'
    series[nombre] = s
panel = pd.DataFrame(series)
assert panel.notna().all().all(), 'Series con fechas diferentes'
display(pd.DataFrame({'n': panel.count(), 'inicio': panel.index.min(), 'fin': panel.index.max()}))
variantes = {}
for nombre, s in series.items():
    variantes[nombre] = {'Nivel': s, 'Δ': s.diff().dropna()}
    if nombre == 'Viajes MBTB':
        variantes[nombre].update({'Δ7': s.diff(7).dropna(), 'ΔΔ7': s.diff().diff(7).dropna()})
    elif nombre in ('Temperatura', 'Humedad'):
        variantes[nombre]['Δ365'] = s.diff(365).dropna()


,n,inicio,fin
Viajes MBTB,1096,2023-01-01,2025-12-31
Temperatura,1096,2023-01-01,2025-12-31
Humedad,1096,2023-01-01,2025-12-31
Precipitación,1096,2023-01-01,2025-12-31


In [2]:
from arch.unitroot import PhillipsPerron
import statsmodels, arch
print('Versiones:', 'pandas', pd.__version__, 'statsmodels', statsmodels.__version__, 'arch', arch.__version__)
filas = []
for nombre, opciones in variantes.items():
    for transformacion, s in opciones.items():
        for especificacion in (['c', 'ct'] if transformacion == 'Nivel' else ['c']):
            a = adfuller(s, regression=especificacion, autolag='AIC')
            pp = PhillipsPerron(s, trend=especificacion, test_type='tau')
            with warnings.catch_warnings(record=True) as avisos:
                warnings.simplefilter('always')
                k = kpss(s, regression=especificacion, nlags='auto')
            for prueba, estadistico, p, rezagos, n, criticos, aviso in [
                ('ADF', a[0], a[1], a[2], a[3], a[4], ''),
                ('PP', pp.stat, pp.pvalue, pp.lags, pp.nobs, pp.critical_values, ''),
                ('KPSS', k[0], k[1], k[2], len(s), k[3], ' | '.join(str(w.message) for w in avisos))]:
                p_texto = f'{p:.4g}'
                if prueba == 'KPSS' and aviso:
                    p_texto = ('≤ ' if p <= .01 else '≥ ') + f'{p:.2f}'
                filas.append(dict(serie=nombre, transformacion=transformacion,
                    especificacion=especificacion, prueba=prueba, n=n,
                    estadistico=estadistico, p_valor=p, p_reportado=p_texto,
                    rezagos_o_banda=rezagos, critico_5=criticos['5%'],
                    rechaza_H0=p < .05, advertencia=aviso))
resultados = pd.DataFrame(filas)
pd.set_option('display.max_rows', 100)
display(resultados.drop(columns=['p_valor', 'advertencia']).round(4))
resultados.to_csv(ROOT / 'puntos/punto4/resultados_pruebas.csv', index=False)


Versiones: pandas 3.0.6 statsmodels 0.15.0 arch 8.0.0


/tmp/ipykernel_3044/1179759575.py:8: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  a = adfuller(s, regression=especificacion, autolag='AIC')
/tmp/ipykernel_3044/1179759575.py:8: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  a = adfuller(s, regression=especificacion, autolag='AIC')
/tmp/ipykernel_3044/1179759575.py:8: FutureWarning: adfuller currently returns a plain tuple

/tmp/ipykernel_3044/1179759575.py:8: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  a = adfuller(s, regression=especificacion, autolag='AIC')
/tmp/ipykernel_3044/1179759575.py:8: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  a = adfuller(s, regression=especificacion, autolag='AIC')
/tmp/ipykernel_3044/1179759575.py:8: FutureWarning: adfuller currently returns a plain tuple

,serie,transformacion,especificacion,prueba,n,estadistico,p_reportado,rezagos_o_banda,critico_5,rechaza_H0
0,Viajes MBTB,Nivel,c,ADF,1075,-2.3674,0.1512,20,-2.8642,False
1,Viajes MBTB,Nivel,c,PP,1095,-23.0105,0,22,-2.8642,True
2,Viajes MBTB,Nivel,c,KPSS,1096,1.5933,≤ 0.01,16,0.4630,True
3,Viajes MBTB,Nivel,ct,ADF,1075,-2.4357,0.3608,20,-3.4146,False
4,Viajes MBTB,Nivel,ct,PP,1095,-23.5647,0,22,-3.4145,True
5,Viajes MBTB,Nivel,ct,KPSS,1096,0.9357,≤ 0.01,15,0.1460,True
6,Viajes MBTB,Δ,c,ADF,1075,-13.0722,1.947e-24,19,-2.8642,True
7,Viajes MBTB,Δ,c,PP,1094,-84.8571,0,22,-2.8642,True
8,Viajes MBTB,Δ,c,KPSS,1095,0.1141,≥ 0.10,95,0.4630,False
9,Viajes MBTB,Δ7,c,ADF,1067,-9.6722,1.268e-16,21,-2.8643,True


## Cómo leer la tabla

- **serie / transformacion:** variable analizada y cambio aplicado. “Nivel” es el dato original; Δ resta el día anterior, Δ7 resta el valor de hace siete días y Δ365 resta el de hace 365 días. ΔΔ7 combina la diferencia diaria y semanal.
- **especificacion:** `c` incluye una constante; `ct` incluye también una tendencia lineal.
- **n:** cantidad de observaciones utilizadas por la prueba.
- **estadistico / critico_5:** resultado numérico de la prueba y valor de referencia al 5%. Para facilitar la lectura se puede usar el p-valor y la decisión de rechazo.
- **p_reportado:** p-valor que se debe interpretar según la hipótesis de cada prueba.
- **rezagos_o_banda:** cantidad de días anteriores que intervienen en el ajuste de la prueba.
- **rechaza_H0:** `True` significa que se rechaza la hipótesis de partida; `False`, que no se rechaza.

KPSS solo calcula p-valores dentro de un intervalo. Por eso **≤ 0,01** significa “0,01 o menor” y **≥ 0,10**, “0,10 o mayor”. Un cero numérico en ADF o PP significa que el p-valor es demasiado pequeño para representarlo con esa precisión.

El archivo `resultados_pruebas.csv` conserva los resultados y las advertencias del cálculo. La siguiente tabla reúne las tres pruebas para facilitar su comparación.

In [3]:
resumen = resultados.pivot(index=['serie', 'transformacion', 'especificacion'], columns='prueba', values='p_valor')
def diagnostico(r):
    if r.ADF < .05 and r.PP < .05 and r.KPSS >= .05:
        return 'Compatible con estacionariedad bajo la especificación'
    if r.ADF >= .05 and r.PP >= .05 and r.KPSS < .05:
        return 'Evidencia conjunta compatible con raíz unitaria'
    return 'Evidencia mixta: revisar especificación y estacionalidad'
resumen['lectura_conjunta'] = resumen.apply(diagnostico, axis=1)
display(resumen)


prueba                                                ADF      KPSS  \
serie         transformacion especificacion                           
Humedad       Nivel          c               2.371817e-06  0.086363   
                             ct              3.317746e-05  0.022067   
              Δ              c               1.691205e-21  0.100000   
              Δ365           c               1.979702e-10  0.010000   
Precipitación Nivel          c               0.000000e+00  0.100000   
                             ct              0.000000e+00  0.100000   
              Δ              c               7.079089e-25  0.100000   
Temperatura   Nivel          c               4.058148e-02  0.100000   
                             ct              1.869369e-01  0.039285   
              Δ              c               2.226409e-27  0.100000   
              Δ365           c               6.658944e-08  0.100000   
Viajes MBTB   Nivel          c               1.511636e-01  0.010000   
                             ct              3.608400e-01  0.010000   
              Δ              c               1.946632e-24  0.100000   
              Δ7             c               1.268277e-16  0.100000   
              ΔΔ7            c               2.408892e-26  0.100000   

prueba                                                 PP  \
serie         transformacion especificacion                 
Humedad       Nivel          c               1.690081e-21   
                             ct              5.820971e-19   
              Δ              c               0.000000e+00   
              Δ365           c               1.689816e-23   
Precipitación Nivel          c               0.000000e+00   
                             ct              0.000000e+00   
              Δ              c               0.000000e+00   
Temperatura   Nivel          c               2.780132e-06   
                             ct              2.911381e-05   
              Δ              c               0.000000e+00   
              Δ365           c               1.977736e-16   
Viajes MBTB   Nivel          c               0.000000e+00   
                             ct              0.000000e+00   
              Δ              c               0.000000e+00   
              Δ7             c               0.000000e+00   
              ΔΔ7            c               0.000000e+00   

prueba                                                                        lectura_conjunta  
serie         transformacion especificacion                                                     
Humedad       Nivel          c               Compatible con estacionariedad bajo la especif...  
                             ct              Evidencia mixta: revisar especificación y esta...  
              Δ              c               Compatible con estacionariedad bajo la especif...  
              Δ365           c               Evidencia mixta: revisar especificación y esta...  
Precipitación Nivel          c               Compatible con estacionariedad bajo la especif...  
                             ct              Compatible con estacionariedad bajo la especif...  
              Δ              c               Compatible con estacionariedad bajo la especif...  
Temperatura   Nivel          c               Compatible con estacionariedad bajo la especif...  
                             ct              Evidencia mixta: revisar especificación y esta...  
              Δ              c               Compatible con estacionariedad bajo la especif...  
              Δ365           c               Compatible con estacionariedad bajo la especif...  
Viajes MBTB   Nivel          c               Evidencia mixta: revisar especificación y esta...  
                             ct              Evidencia mixta: revisar especificación y esta...  
              Δ              c               Compatible con estacionariedad bajo la especif...  
              Δ7             c               Compatible con estacionar

## Conclusiones por serie

### Viajes en bicicleta

Con la opción de media fija, ADF no rechaza raíz unitaria (p≈0,151) y KPSS rechaza estacionariedad (p≤0,01). Sin embargo, PP sí rechaza raíz unitaria (p<0,001). Al agregar una tendencia ocurre el mismo desacuerdo: ADF no rechaza (p≈0,361), PP rechaza y KPSS rechaza estacionariedad alrededor de la tendencia.

**¿Qué significa?** Las pruebas no coinciden. No se puede afirmar con certeza que la serie tenga una raíz unitaria. La diferencia diaria y la semanal son alternativas para estudiar: en ambas, ADF rechaza raíz unitaria y KPSS no rechaza estacionariedad. Esto no significa que haya que aplicar las dos juntas. El gráfico del punto 3 también invita a revisar si la diferencia semanal agrega una relación negativa innecesaria.

### Temperatura

Con media fija, ADF rechaza raíz unitaria (p≈0,0406), aunque está cerca del límite de 0,05. PP también rechaza y KPSS no rechaza estacionariedad (p≥0,10). Con tendencia, las pruebas dejan de coincidir: ADF no rechaza raíz unitaria (p≈0,187), PP la rechaza y KPSS rechaza estacionariedad alrededor de la tendencia (p≈0,0393).

**¿Qué significa?** La conclusión depende de la opción utilizada. No corresponde decidir automáticamente que hay que restar el valor del día anterior. El ciclo anual sigue siendo importante, aunque no se encuentre evidencia clara de una raíz unitaria. Restar el valor de hace un año es una opción para comparar, no una necesidad demostrada.

### Humedad

Con media fija, ADF y PP rechazan raíz unitaria y KPSS no rechaza estacionariedad (p≈0,0864). Con tendencia, ADF y PP siguen rechazando raíz unitaria, pero KPSS rechaza estacionariedad alrededor de la tendencia (p≈0,0221).

Después de aplicar la diferencia anual, ADF rechaza raíz unitaria, pero KPSS rechaza estacionariedad (p≤0,01).

**¿Qué significa?** La diferencia anual propuesta en el punto 2 no queda confirmada. No alcanza con restar el valor de hace un año para asegurar que la serie sea estacionaria. Hace falta revisar cómo cambia el patrón estacional y qué opción describe mejor los datos.

### Precipitación

ADF y PP rechazan raíz unitaria y KPSS no rechaza estacionariedad, tanto con media fija como con tendencia.

**¿Qué significa?** Las tres pruebas coinciden en un resultado compatible con conservar la serie original. Esto también concuerda con la relación de corta duración observada en el punto 3. Los días sin lluvia y los picos de precipitación no son, por sí solos, una razón para diferenciar.

## Relación con el punto 2

Los resultados permiten ajustar las primeras impresiones de los gráficos. **Una serie con un ciclo anual no necesita necesariamente una diferencia anual.** Del mismo modo, que una serie diferenciada oscile alrededor de cero no demuestra que sea estacionaria.

Además, diferenciar reduce los datos disponibles. La diferencia anual deja 731 observaciones de las 1096 originales. Las pruebas se aplican a los datos disponibles en cada caso; sus p-valores no sirven para decidir qué modelo pronostica mejor. Esa decisión requiere comparar modelos, comprobar sus errores y evaluar sus pronósticos en los puntos siguientes.

## Referencias y herramientas

Statsmodels developers. (s. f.). *Augmented Dickey–Fuller test*. https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.adfuller.html

Statsmodels developers. (s. f.). *Kwiatkowski–Phillips–Schmidt–Shin test*. https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.kpss.html

Sheppard, K. (s. f.). *Phillips–Perron unit root test*. arch. https://bashtage.github.io/arch/unitroot/generated/arch.unitroot.PhillipsPerron.html

Se utilizan pandas y NumPy para organizar los datos, Matplotlib para los gráficos, statsmodels para ADF y KPSS, y arch para PP. Los datos provienen de los archivos CSV del proyecto, cuya procedencia se describe en el punto 1.
